## Simple Rag Demo for this Project

### Import Necessary Libraries

In [ ]:
import os
from dotenv import load_dotenv

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import JinaEmbeddings

In [4]:
load_dotenv()

groq_key = os.getenv("GROQ_API_KEY")
jina_key = os.getenv("JINA_API_KEY")

print("Env variables loaded")

Env variables loaded


### Loading our Data

In [6]:
DATA_FILE_PATH = os.path.join("data", "hr_policy.txt")
print(DATA_FILE_PATH)

data/hr_policy.txt


In [7]:
loader = TextLoader(DATA_FILE_PATH, encoding="utf-8")
documents = loader.load()

print(f"Number of documents loaded: {len(documents)}")
print(f"Total characters in document: {len(documents[0].page_content)}")

Number of documents loaded: 1
Total characters in document: 2598


In [11]:
print(documents[0].metadata)


{'source': 'data/hr_policy.txt'}


### Splitting the Data

In [17]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)


print(len(chunks))

9


In [18]:
print(chunks[0])

page_content='COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)' metadata={'source': 'data/hr_policy.txt'}


In [19]:
print(chunks[4].page_content)

4. NOTICE PERIOD
Employees who wish to resign must serve a notice period of 30 days.
During the probation period, the notice period is reduced to 15 days.
The company may waive the notice period at its discretion, with full and final settlement
processed within 45 days of the last working day.


In [20]:
print(chunks[5].page_content)

5. REIMBURSEMENT POLICY
Employees can claim reimbursement for approved business expenses such as travel,
client meals, and internet bills used for official work.
All reimbursement claims must be submitted with valid bills within 30 days of the expense.
Claims are processed within 10 working days after approval from the reporting manager.


In [21]:
print(chunks[6].page_content)

6. CODE OF CONDUCT
Employees are expected to maintain professionalism and respect in the workplace.
Harassment, discrimination, or any form of workplace misconduct will not be tolerated
and may result in disciplinary action, including termination.
All employees must complete an annual Code of Conduct training.


### Embedding the Data

In [23]:
from langchain_community.embeddings import JinaEmbeddings

embeddings_model = JinaEmbeddings(model_name="jina-embeddings-v2-base-en")


In [24]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(chunks, embeddings_model)

In [25]:
vector_store.index.ntotal

9

In [27]:
# Similarity search

test_query = "How many sick leaves employees get"

top_matches = vector_store.similarity_search(test_query, k=3)

print(f"Query: {test_query}\n")
for i,match in enumerate(top_matches,start=1):
    print(f"--- Match {i} ---")
    print(match.page_content)
    print()

Query: How many sick leaves employees get

--- Match 1 ---
1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

--- Match 2 ---
7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.

--- Match 3 ---
3. PROBATION PERIOD
All new employees undergo a probation period of 3 months from their date of joining.
During probation, employees are not eligible for paid leave, but may take unpaid leave
in case of emergencies, subject to manager a

### Tool for Searching the Data

In [28]:
retriever = vector_store.as_retriever(search_kwargs={"k":3})

def search_hr_policy(question: str)->str:
    """
    Search the HR policy document for information about leave, work from home,
    probation, notice period, reimbursement, code of conduct, holidays, or exit process.
    
    """
    matching_chunks = retriever.invoke(question)
    return "\n\n".join(chunk.page_content for chunk in matching_chunks)

### Data Retrieval and Answer Generation

In [29]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model = "openai/gpt-oss-120b",
    temperature=0.5  # creativity 
)

llm.model_name

'openai/gpt-oss-120b'

In [30]:
tr = llm.invoke("Hey what is the leave policy")

In [31]:
tr.content

"Sure thing! Could you let me know which organization or country's leave policy you’re interested in? That’ll help me give you the most relevant information."

In [32]:
from langchain.agents import create_agent 

hr_assistant = create_agent(
    model = llm,
    tools=[search_hr_policy],
    system_prompt= """ 
    
    You are a friendly HR assistant working for Acme Crop. 
    Always use the search_hr_policy tool to look up 
    facts before answering. 
    If the answer isn't in the search results, say you don't know "
    instead of guessing."
    """
)

print("HR assistant agent is ready to answer questions!")

HR assistant agent is ready to answer questions!


In [33]:
def ask_hr_assistant(question: str) -> str:
    """Send a question to the RAG agent and print a nicely formatted answer."""
    print("=" * 60)
    print("QUESTION:", question)
    print("-" * 60)

    response = hr_assistant.invoke({"messages": [{"role": "user", "content": question}]})
    answer = response["messages"][-1].content

    print("ANSWER:", answer)
    print("=" * 60)
    print()
    return answer

In [34]:
response = hr_assistant.invoke(
    {
        "messages":[
            {
                "role":"user",
                "content": "tell me which org you work for"
            }
        ]
    }
)


In [35]:
response 

{'messages': [HumanMessage(content='tell me which org you work for', additional_kwargs={}, response_metadata={}, id='c93e0dfd-ec78-400b-a865-1d145712e38f'),
  AIMessage(content='I’m the friendly HR assistant for **Acme\u202fCrop**. How can I help you today?', additional_kwargs={'reasoning_content': 'User asks: "tell me which org you work for". As HR assistant for Acme Crop. Should answer accordingly. No need to search policy. Provide answer.'}, response_metadata={'token_usage': {'completion_tokens': 64, 'prompt_tokens': 208, 'total_tokens': 272, 'completion_time': 0.136161892, 'completion_tokens_details': {'reasoning_tokens': 34}, 'prompt_time': 0.055025923, 'prompt_tokens_details': None, 'queue_time': 0.352302576, 'total_time': 0.191187815}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_4200b3f836', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a09bd9-018a-7943-b5f5-39b72eee6831-0', tool_calls=[], inva

In [36]:
response["messages"][-2].content

'tell me which org you work for'

In [37]:
print(response["messages"][-1].content)

I’m the friendly HR assistant for **Acme Crop**. How can I help you today?
